# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"Dataset name: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id and fields for inspection
record_sets = dataset.metadata.record_sets
print(f"Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}, name: {rs.get('name','(no name)')}")
    fields = rs.get('fields', [])
    print("  Fields:")
    for field in fields:
        # many times a field is an object or {'@id': '<id>'}
        if isinstance(field, dict) and '@id' in field:
            print(f"    Field @id: {field['@id']}")
        elif isinstance(field, str):
            print(f"    Field @id: {field}")
        elif isinstance(field, dict) and 'name' in field:
            print(f"    Field: {field['name']}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, there should be a main data record set.
# We'll extract a list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Display all available record set IDs
print("Record set @ids:", record_set_ids)

# Load all record sets into dataframes
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    print(f"Loaded {len(records)} records from record set {record_set_id}")
    # If the set isn't empty, create the dataframe
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# For subsequent analysis, pick the main record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]  # adjust if needed
    if main_record_set_id in dataframes:
        print(f"\nColumns in main record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
        dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field (e.g., GAD-7 or PHQ-9 score field) for demo
# Find all numeric-like columns
import numpy as np

df = dataframes[main_record_set_id]

# Try to auto-discover a plausible score field
candidate_cols = [c for c in df.columns if any(x in c.lower() for x in ['gad', 'phq', 'score', 'psq'])]
print(f"Candidate numeric fields: {candidate_cols}")
if candidate_cols:
    numeric_field = candidate_cols[0]
    print(f"Using {numeric_field} as the numeric field for analysis.")
    # Ensure numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    
    # Filtering: keep only scores greater than a threshold (e.g., 10, for moderate symptoms)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)} records):\n")
    print(filtered_df[[numeric_field]].head())

    # Normalization (Z-score)
    filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

    # Group by field (e.g., 'level_of_education' if available)
    group_field_candidates = [c for c in df.columns if 'education' in c.lower() or 'gender' in c.lower()]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"\nGrouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if candidate_cols:
    # Histogram of main numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Box plot by education/gender if available
    if group_field_candidates:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Croissant-based dataset was successfully loaded using `mlcroissant` and explored via its record set(s) and fields.
- Numeric mental health assessment fields, such as GAD-7, PHQ-9, or PSQ, were identified for basic EDA and visualized for distribution.
- Basic filtering and normalization show how to preprocess Croissant datasets for downstream machine learning and statistical analysis.
- Grouped summary statistics by demographic fields (such as education or gender, if present) can be further extended for domain-specific insights.

For additional documentation and advanced `mlcroissant` features, see: https://mlcommons.github.io/croissant/